In [ ]:
# precompute_esmc_from_txt.py
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# 设置缓存目录（可选，避免路径过长问题）
os.environ["HF_HOME"] = "E:/huggingface/hub"
import json
import re
import torch
import numpy as np
from tqdm import tqdm
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig


def read_sequences_from_txt(txt_path):
    """
    读取 [Cancer] sequence 格式的文本文件，返回所有序列（去重）
    """
    seqs = []
    with open(txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            match = re.match(r'\[(.+?)\]\s+(.+)', line)
            if match:
                seq = match.group(2).strip().upper()
                seq = re.sub(r'[^A-Z]', '', seq)
                if len(seq) >= 5:
                    seqs.append(seq)
    # 去重（可能同一序列对应多种癌症，但这里我们只关心序列本身）
    unique_seqs = list(set(seqs))
    print(f"读取到 {len(seqs)} 条记录，去重后 {len(unique_seqs)} 条唯一序列")
    return unique_seqs

def precompute_embeddings(txt_path, output_path, model_size="600m", device="cuda", batch_size=8):
    """
    从 txt 文件提取所有序列的 ESMC 残基级嵌入
    """
    print(f"加载 ESMC {model_size} 模型...")
    model = ESMC.from_pretrained(f"esmc_{model_size}").to(device)
    model.eval()
    print("模型加载完成")

    seqs = read_sequences_from_txt(txt_path)
    print(f"开始处理 {len(seqs)} 条唯一序列...")

    embeddings_dict = {}
    total = len(seqs)

    for i in tqdm(range(0, total, batch_size), desc="提取嵌入"):
        batch_seqs = seqs[i:i+batch_size]
        batch_proteins = [ESMProtein(sequence=s) for s in batch_seqs]
        try:
            protein_tensors = [model.encode(p) for p in batch_proteins]
            outputs = [
                model.logits(t, LogitsConfig(sequence=True, return_embeddings=True))
                for t in protein_tensors
            ]
            for out, seq in zip(outputs, batch_seqs):
                # out.embeddings shape: (1, L, 1152)
                residue_emb = out.embeddings[0]  # (L, 1152)
                L = len(seq)
                # 确保长度与序列一致
                if residue_emb.shape[0] != L:
                    print(f"警告: 序列 {seq[:10]}... 长度 {L} 与嵌入长度 {residue_emb.shape[0]} 不一致，进行截断/填充")
                    if residue_emb.shape[0] > L:
                        residue_emb = residue_emb[:L]
                    else:
                        pad = torch.zeros((L - residue_emb.shape[0], residue_emb.shape[1]), device=residue_emb.device)
                        residue_emb = torch.cat([residue_emb, pad], dim=0)
                embeddings_dict[seq] = residue_emb.cpu().numpy().tolist()
        except Exception as e:
            print(f"批次处理失败: {e}, 跳过该批次")
            continue

    # 保存为 JSON
    print(f"保存嵌入到 {output_path}...")
    with open(output_path, 'w') as f:
        json.dump(embeddings_dict, f)

    print(f"完成！共保存 {len(embeddings_dict)} 条序列的嵌入")
    
    # 简单验证
    with open(output_path, 'r') as f:
        saved = json.load(f)
    empty = sum(1 for v in saved.values() if len(v) == 0)
    print(f"空嵌入（长度为0）: {empty} 条")
    if len(saved) > 0:
        first_key = list(saved.keys())[0]
        print(f"示例嵌入形状: 序列 '{first_key[:10]}...' 的嵌入形状: {len(saved[first_key])} x {len(saved[first_key][0])}")

if __name__ == "__main__":
    # 配置参数
    txt_path = "data/processed/prefix_training_data.txt"
    output_path = "data/esmc_embeddings_top6.json"
    model_size = "600m"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    precompute_embeddings(txt_path, output_path, model_size, device, batch_size=8)
    

加载 ESMC 600m 模型...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


esmc_600m_2024_12_v0.pth:  46%|####6     | 1.97G/4.27G [00:00<?, ?B/s]

模型加载完成
读取到 2239 条记录，去重后 922 条唯一序列
开始处理 922 条唯一序列...


...

提取嵌入:  99%|█████████▉| 115/116 [01:44<00:00,  1.14it/s]

警告: 序列 PRFWEYWLAL... 长度 12 与嵌入长度 14 不一致，进行截断/填充
警告: 序列 GLFKVIKKVA... 长度 16 与嵌入长度 18 不一致，进行截断/填充
警告: 序列 GTGLPMSERR... 长度 17 与嵌入长度 19 不一致，进行截断/填充
警告: 序列 VAKLLAKLAK... 长度 13 与嵌入长度 15 不一致，进行截断/填充
警告: 序列 FRRFFKWFRR... 长度 15 与嵌入长度 17 不一致，进行截断/填充
警告: 序列 EFLDCFQKF... 长度 9 与嵌入长度 11 不一致，进行截断/填充
警告: 序列 PAWRKAFRWA... 长度 18 与嵌入长度 20 不一致，进行截断/填充
警告: 序列 FAKLLAKLAK... 长度 13 与嵌入长度 15 不一致，进行截断/填充


提取嵌入: 100%|██████████| 116/116 [01:44<00:00,  1.11it/s]

警告: 序列 VKRFKKFFRK... 长度 15 与嵌入长度 17 不一致，进行截断/填充
警告: 序列 GKLRLIKKLW... 长度 21 与嵌入长度 23 不一致，进行截断/填充
保存嵌入到 data/esmc_embeddings_top6.json...


完成！共保存 922 条序列的嵌入
空嵌入（长度为0）: 0 条
示例嵌入形状: 序列 'GRKKRRQRRR...' 的嵌入形状: 10 x 1152


In [9]:
# -*- coding: utf-8 -*-
import sys
import os
import json

# ========== 添加项目根目录到 sys.path（兼容交互式环境） ==========
try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()
sys.path.insert(0, script_dir)

import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ========== 导入项目自己的特征提取模块 ==========
from util.embed.embedding import sequence_embedding
from util.embed.sequence import ESMC_EMBEDDINGS

# ==================== 配置 ====================
REAL_DATA_FILE = "data/processed/prefix_training_data.txt"
ESMC_JSON_PATH = "data/esmc_embeddings_new.json"
TSNE_PERPLEXITY = 10          # 样本增多，适度提高
TSNE_RANDOM_STATE = 42
PCA_COMPONENTS = 150          # 保留更多主成分（原200，但样本量2239，可适当）
# ==============================================

# ====== 手动加载 ESMC 嵌入缓存 ======
print("加载 ESMC 嵌入缓存...")
try:
    with open(ESMC_JSON_PATH, 'r') as f:
        ESMC_EMBEDDINGS = json.load(f)
    print(f"✅ 成功加载 ESMC 嵌入: {len(ESMC_EMBEDDINGS)} 条序列")
except Exception as e:
    print(f"❌ ESMC 嵌入加载失败: {e}")
    ESMC_EMBEDDINGS = {}
# ===================================

# ---------- 全局绘图风格 ----------
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 8,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.linewidth': 1.2,
    'axes.edgecolor': 'black',
    'axes.grid': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'legend.frameon': True,
    'legend.edgecolor': 'gray',
    'legend.facecolor': 'white',
    'legend.framealpha': 0.9,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})

# ==================== 工具函数 ====================
def read_peptide_file(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            match = re.match(r'\[(.+?)\]\s+(.+)', line)
            if match:
                cancer = match.group(1).strip()
                seq = match.group(2).strip().upper()
                seq = re.sub(r'[^A-Z]', '', seq)
                if len(seq) >= 5:
                    data.append({'Sequence': seq, 'norm_cancer': cancer})
    return pd.DataFrame(data)

def get_model_features(seqs):
    """使用模型的 sequence_embedding 函数提取特征 (2072维)"""
    features = []
    print(f"正在为 {len(seqs)} 条序列提取模型特征 (2072 维)...")
    for i, seq in enumerate(seqs):
        if i % 50 == 0:
            print(f"  进度: {i}/{len(seqs)}")
        try:
            feat = sequence_embedding(fasta=seq)
            seq_feat = feat.mean(axis=0)
            features.append(seq_feat)
        except Exception as e:
            print(f"  序列 {seq[:10]}... 特征提取失败: {e}")
            features.append(np.zeros(2072))
    return np.array(features)

def reduce_tsne(X, n_samples):
    if n_samples > 200:
        pca_comp = min(PCA_COMPONENTS, X.shape[1], n_samples - 1)
        pca = PCA(n_components=pca_comp, random_state=TSNE_RANDOM_STATE)
        X_pca = pca.fit_transform(X)
        perp = min(TSNE_PERPLEXITY, n_samples - 1)
        tsne = TSNE(n_components=2, perplexity=perp, random_state=TSNE_RANDOM_STATE, max_iter=1000)
        return tsne.fit_transform(X_pca)
    else:
        perp = min(TSNE_PERPLEXITY, n_samples - 1)
        tsne = TSNE(n_components=2, perplexity=perp, random_state=TSNE_RANDOM_STATE, max_iter=1000)
        return tsne.fit_transform(X)

# ==================== 主流程 ====================
print("\n" + "="*60)
print("1. 读取真实数据 (仅 Top 6 癌症)...")
df_real = read_peptide_file(REAL_DATA_FILE)
print(f"原始样本数: {len(df_real)}")
print("各癌症样本数:\n", df_real['norm_cancer'].value_counts())

# ====== 【修改点】跳过特异序列过滤，直接使用全部数据 ======
print("\n2. 使用全部样本（不进行特异性过滤）...")
# 注释掉以下过滤代码
# seq_to_cancers = {}
# for _, row in df_real.iterrows():
#     seq = row['Sequence']
#     cancer = row['norm_cancer']
#     seq_to_cancers.setdefault(seq, set()).add(cancer)
# specific_seqs_list = [seq for seq, cancers in seq_to_cancers.items() if len(cancers) == 1]
# df_specific = df_real[df_real['Sequence'].isin(specific_seqs_list)].copy()
# print(f"特异序列总数: {len(df_specific)}")
# print("各癌症特异序列数量:\n", df_specific['norm_cancer'].value_counts())
# df_real = df_specific

# 直接使用 df_real
print(f"用于绘图的样本数: {len(df_real)}")

real_seqs = df_real['Sequence'].tolist()
real_cancers = df_real['norm_cancer'].tolist()
cancer_types = sorted(df_real['norm_cancer'].unique())
print(f"癌症类别数: {len(cancer_types)}")

print("\n3. 提取模型特征 (2072 维)...")
X_feat = get_model_features(real_seqs)
print(f"特征矩阵形状: {X_feat.shape}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_feat)

print("\n4. 执行 t-SNE...")
X_tsne = reduce_tsne(X_scaled, len(X_scaled))
print("   t-SNE 完成")

cmap = plt.cm.get_cmap('tab20', len(cancer_types))
cancer_colors = {c: cmap(i) for i, c in enumerate(cancer_types)}

print("\n5. 绘图...")
plt.figure(figsize=(12, 10))
ax = plt.gca()
for cancer in cancer_types:
    idx = [i for i, c in enumerate(real_cancers) if c == cancer]
    if idx:
        ax.scatter(
            X_tsne[idx, 0], X_tsne[idx, 1],
            c=[cancer_colors[cancer]],
            label=cancer,
            alpha=0.6,
            s=30,
            marker='o',
            edgecolors='none'
        )
ax.set_title(
    f't-SNE of All Sequences (Model Features 2072-dim, Top 6 Cancers)\n'
    f'{len(real_seqs)} sequences, {len(cancer_types)} cancer types',
    fontweight='bold', pad=15
)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlim(X_tsne[:, 0].min() - 1, X_tsne[:, 0].max() + 1)
ax.set_ylim(X_tsne[:, 1].min() - 1, X_tsne[:, 1].max() + 1)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True,
          edgecolor='black', facecolor='white', framealpha=0.9, fontsize=8, ncol=1)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2)
    spine.set_color('black')
plt.tight_layout()
plt.savefig('tsne_all_model_features_top6.png', dpi=300, bbox_inches='tight')
plt.close()
print("已保存: tsne_all_model_features_top6.png")

print("\n" + "="*60)
print(f"{'癌症':<20s} {'样本数':>8s}")
print("-"*30)
for cancer in cancer_types:
    count = sum(1 for c in real_cancers if c == cancer)
    print(f"{cancer:<20s} {count:>8d}")

print("\n✅ 特征组成 (2072 维):")
print("   - One-hot: 20 维")
print("   - BLOSUM: 20 维")
print("   - PAM: 20 维")
print("   - 疏水性: 5 维")
print("   - ESMC: 1152 维")
print("   - AAC: 20 维")
print("   - AARPC: 20 维")
print("   - CKSAPP: 800 维")
print("   - 总计: 2052 + 20 = 2072 维")

print("\n✅ 完成！生成 t-SNE 图（全部序列 + 模型完整特征）。")

加载 ESMC 嵌入缓存...
❌ ESMC 嵌入加载失败: [Errno 2] No such file or directory: 'data/esmc_embeddings_new.json'

1. 读取真实数据 (仅 Top 6 癌症)...
原始样本数: 2813
各癌症样本数:
 norm_cancer
Breast      535
Lung        423
Colon       408
Cervix      345
Skin        306
Prostate    284
Blood       243
Liver       153
Brain       116
Name: count, dtype: int64

2. 使用全部样本（不进行特异性过滤）...
用于绘图的样本数: 2813
癌症类别数: 9

3. 提取模型特征 (2072 维)...
正在为 2813 条序列提取模型特征 (2072 维)...
  进度: 0/2813
  进度: 50/2813
  进度: 100/2813
  进度: 150/2813
  进度: 200/2813
  进度: 250/2813
  进度: 300/2813
  进度: 350/2813
  进度: 400/2813
  进度: 450/2813
  进度: 500/2813
  进度: 550/2813
  进度: 600/2813
  进度: 650/2813
  进度: 700/2813
  进度: 750/2813
  进度: 800/2813
  进度: 850/2813
  进度: 900/2813
  进度: 950/2813
  进度: 1000/2813
  进度: 1050/2813
  进度: 1100/2813
  进度: 1150/2813
  进度: 1200/2813
  进度: 1250/2813
  进度: 1300/2813
  进度: 1350/2813
  进度: 1400/2813
  进度: 1450/2813
  进度: 1500/2813
  进度: 1550/2813
  进度: 1600/2813
  进度: 1650/2813
  进度: 1700/2813
  进度: 1750/2813
  进度: 1800/2813

In [5]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import json
import numpy as np
from tqdm import tqdm
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig
from Bio import SeqIO
protein = ESMProtein(sequence="AAAAA")
client = ESMC.from_pretrained("esmc_600m").to("cuda") # or "cpu"
#protein_tensor = client.encode(protein)
#logits_output = client.logits(
#   protein_tensor, LogitsConfig(sequence=True, return_embeddings=True)
#)
#print(logits_output.logits, logits_output.embeddings)

LocalEntryNotFoundError: An error happened while trying to locate the files on the Hub and we cannot find the appropriate snapshot folder for the specified revision on the local disk. Please check your internet connection and try again.

In [ ]:
#验证

import json
import numpy as np

# 加载嵌入文件
with open("data/esmc_embeddings4.json", 'r') as f:
    embeddings = json.load(f)


In [ ]:
# 检查所有序列的嵌入
empty_count = 0
total_count = 0

for seq, embedding in embeddings.items():
    total_count += 1
    if len(embedding) == 0:
        print(f"Empty embedding for {seq}")
        empty_count += 1
    else:
        embedding_array = np.array(embedding)
        print(f"{seq}: shape {embedding_array.shape}, sample values {embedding_array[0][:5]}")

print(f"\nSummary: {empty_count} empty embeddings out of {total_count} sequences")

ACP164valid_pos_1: shape (13, 1152), sample values [-0.00988388  0.00095824 -0.00473444 -0.00619367 -0.00014632]
ACP164valid_pos_2: shape (16, 1152), sample values [-0.01100189 -0.00140806 -0.01828986 -0.00819758  0.00641512]
ACP164valid_pos_3: shape (30, 1152), sample values [-0.01591619  0.01502713 -0.01242523 -0.00795439  0.00537973]
ACP164valid_pos_4: shape (26, 1152), sample values [-0.00919525  0.00348636  0.00015209 -0.00812692  0.00262657]
ACP164valid_pos_5: shape (28, 1152), sample values [-0.00931274  0.01331421 -0.00974715 -0.00642234 -0.00017902]
ACP164valid_pos_7: shape (27, 1152), sample values [-0.00247382  0.01382646 -0.0103598  -0.00963737  0.01148649]
ACP164valid_pos_8: shape (23, 1152), sample values [-0.01265044  0.00786505 -0.01178582 -0.00475433  0.00194575]
ACP164valid_pos_9: shape (13, 1152), sample values [-0.0068051   0.00179482 -0.00781735 -0.01049668  0.00905931]
ACP164valid_pos_10: shape (17, 1152), sample values [-0.00972544  0.00385477 -0.00693402 -0.0084

In [2]:
import os
import json
import numpy as np
from tqdm import tqdm
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig
from Bio import SeqIO

def precompute_embeddings(fasta_path, output_path, model_size="600m", device="cuda"):
    """
    预计算所有序列的ESMC嵌入
    
    参数:
        fasta_path: FASTA文件路径
        output_path: 输出JSON文件路径
        model_size: ESMC模型大小 ("300m"或"600m")
        device: 计算设备 ("cuda"或"cpu")
    """
    # 加载模型
    print(f"Loading ESMC {model_size} model...")
    client = ESMC.from_pretrained(f"esmc_{model_size}").to(device)
    print("Model loaded.")
    
    # 读取所有序列
    sequences = []
    for record in SeqIO.parse(fasta_path, "fasta"):
        sequences.append({
            "id": record.id,
            "sequence": str(record.seq)
        })
    
    # 存储嵌入结果
    embeddings_dict = {}
    
    # 分批处理序列
    batch_size = 1  # 改为1，确保每个序列独立处理
    for i in tqdm(range(0, len(sequences), batch_size), desc="Processing sequences"):
        batch = sequences[i:i+batch_size]
        
        try:
            # 创建蛋白质对象
            proteins = [ESMProtein(sequence=seq["sequence"]) for seq in batch]
            
            # 编码蛋白质
            protein_tensors = [client.encode(protein) for protein in proteins]
            
            # 获取logits和嵌入
            logits_outputs = [
                client.logits(
                    tensor, 
                    LogitsConfig(sequence=True, return_embeddings=True)
                ) for tensor in protein_tensors
            ]
            
            # 提取嵌入并存储
            for j, output in enumerate(logits_outputs):
                seq_id = batch[j]["id"]
                seq = batch[j]["sequence"]
                
                # 调试信息
                print(f"Processing: {seq_id}")
                print(f"Sequence: {seq}")
                print(f"Embeddings shape: {output.embeddings.shape}")
                
                # 关键修改：正确提取残基级嵌入
                # 根据你的图片，嵌入形状是 [1, L, 1152]，其中 L 是序列长度
                # 我们需要去掉批次维度，得到 [L, 1152]
                residue_embeddings = output.embeddings[0]  # 去掉批次维度
                
                # 确保嵌入形状与序列长度匹配
                if residue_embeddings.shape[0] != len(seq):
                    print(f"Warning: Embedding length {residue_embeddings.shape[0]} doesn't match sequence length {len(seq)}")
                    # 尝试调整：取前len(seq)个嵌入
                    residue_embeddings = residue_embeddings[:len(seq)]
                
                # 转换为列表并存储
                embeddings_dict[seq] = residue_embeddings.cpu().numpy().tolist()
                
                # 验证嵌入不是空的
                if len(embeddings_dict[seq]) == 0:
                    print(f"Error: Empty embedding for {seq}")
                else:
                    print(f"Success: Extracted {len(embeddings_dict[seq])} residues for {seq}")
                
        except Exception as e:
            print(f"Error processing {seq}: {e}")
            continue
    
    # 保存结果
    with open(output_path, 'w') as f:
        json.dump(embeddings_dict, f)
    print(f"Embeddings saved to {output_path}")
    
    # 验证保存的文件
    with open(output_path, 'r') as f:
        saved_data = json.load(f)
    
    # 检查是否有空值
    empty_count = sum(1 for v in saved_data.values() if len(v) == 0)
    print(f"Found {empty_count} empty embeddings in saved file")

In [3]:
if __name__ == "__main__":
    # 配置参数
    fasta_path = "data/source/fasta/ACP.fasta"
    output_path = "data/esmc_embeddings4.json"
    model_size = "600m"
    device = "cuda"
    
    precompute_embeddings(fasta_path, output_path, model_size, device)

Loading ESMC 600m model...
Model loaded.


Processing sequences:   0%|          | 3/2512 [00:02<24:36,  1.70it/s]  

Processing: cancerppd2_1033
Sequence: FLPLLAGLAANFLPTIICKISYKC
Embeddings shape: torch.Size([1, 26, 1152])
Success: Extracted 24 residues for FLPLLAGLAANFLPTIICKISYKC
Processing: cancerppd2_1034
Sequence: LLAGLAANFLPTIICKISYKC
Embeddings shape: torch.Size([1, 23, 1152])
Success: Extracted 21 residues for LLAGLAANFLPTIICKISYKC
Processing: cancerppd2_1035
Sequence: FAGLAANFLPTIICKISYKC
Embeddings shape: torch.Size([1, 22, 1152])
Success: Extracted 20 residues for FAGLAANFLPTIICKISYKC
Processing: cancerppd2_1036
Sequence: FLKLLKKLAAKFLPTIICKISYKC
Embeddings shape: torch.Size([1, 26, 1152])
Success: Extracted 24 residues for FLKLLKKLAAKFLPTIICKISYKC


Processing sequences:   0%|          | 8/2512 [00:02<07:01,  5.95it/s]

Processing: cancerppd2_1038
Sequence: FLKLLKKLAAKLF
Embeddings shape: torch.Size([1, 15, 1152])
Success: Extracted 13 residues for FLKLLKKLAAKLF
Processing: cancerppd2_1039
Sequence: FLGALFKALSKLL
Embeddings shape: torch.Size([1, 15, 1152])
Success: Extracted 13 residues for FLGALFKALSKLL
Processing: cancerppd2_1040
Sequence: FLKLLAGLLKNFA
Embeddings shape: torch.Size([1, 15, 1152])
Success: Extracted 13 residues for FLKLLAGLLKNFA
Processing: cancerppd2_1410
Sequence: AIGKFLHSAKKFGKAFVGEIMNS
Embeddings shape: torch.Size([1, 25, 1152])
Success: Extracted 23 residues for AIGKFLHSAKKFGKAFVGEIMNS
Processing: cancerppd2_1409
Sequence: GIGKFLHSAKKFAKAFVAEIMNS
Embeddings shape: torch.Size([1, 25, 1152])
Success: Extracted 23 residues for GIGKFLHSAKKFAKAFVAEIMNS
Processing: cancerppd2_1110
Sequence: RAGLQFPVGRLLRRLLRRLLR
Embeddings shape: torch.Size([1, 23, 1152])
Success: Extracted 21 residues for RAGLQFPVGRLLRRLLRRLLR


...

Processing sequences: 100%|█████████▉| 2507/2512 [01:36<00:00, 27.95it/s]

Processing: MLACP20Training_pos_1072
Sequence: MNFQQRLQSLWTLARPFCPPLLATASQMQMVVLPCLGFTLLLWSQVSG
Embeddings shape: torch.Size([1, 50, 1152])
Success: Extracted 48 residues for MNFQQRLQSLWTLARPFCPPLLATASQMQMVVLPCLGFTLLLWSQVSG
Processing: MLACP20Training_pos_1073
Sequence: TGALVQQQDP
Embeddings shape: torch.Size([1, 12, 1152])
Success: Extracted 10 residues for TGALVQQQDP
Processing: MLACP20Training_pos_1074
Sequence: EKSSRPEFYKVILGAHEEYIRG
Embeddings shape: torch.Size([1, 24, 1152])
Success: Extracted 22 residues for EKSSRPEFYKVILGAHEEYIRG
Processing: MLACP20Training_pos_1077
Sequence: QQMNQKDFLSLIVS
Embeddings shape: torch.Size([1, 16, 1152])
Success: Extracted 14 residues for QQMNQKDFLSLIVS
Processing: MLACP20Training_pos_1078
Sequence: SPWDICSVTCGGGVQKRSR
Embeddings shape: torch.Size([1, 21, 1152])
Success: Extracted 19 residues for SPWDICSVTCGGGVQKRSR


Processing sequences: 100%|██████████| 2512/2512 [01:36<00:00, 26.05it/s]

Processing: MLACP20Training_pos_1080
Sequence: SKWSECSRTCGGGVKFQER
Embeddings shape: torch.Size([1, 21, 1152])
Success: Extracted 19 residues for SKWSECSRTCGGGVKFQER
Processing: MLACP20Training_pos_1081
Sequence: SPWSSCSVTCGDGVITRIR
Embeddings shape: torch.Size([1, 21, 1152])
Success: Extracted 19 residues for SPWSSCSVTCGDGVITRIR
Processing: MLACP20Training_pos_1082
Sequence: SPSTHPNEGLEENYCRNPDN
Embeddings shape: torch.Size([1, 22, 1152])
Success: Extracted 20 residues for SPSTHPNEGLEENYCRNPDN
Processing: MLACP20Training_pos_1084
Sequence: HGLGHGHEQQHGLGHGHKFKLDDDLEHQGGHVLD
Embeddings shape: torch.Size([1, 36, 1152])
Success: Extracted 34 residues for HGLGHGHEQQHGLGHGHKFKLDDDLEHQGGHVLD


Embeddings saved to data/esmc_embeddings4.json
Found 0 empty embeddings in saved file


In [ ]:
#验证esmc嵌入的维度信息

import json
import numpy as np

# 加载嵌入文件
with open("data/esmc_embeddings4.json", 'r') as f:
    embeddings = json.load(f)

# 检查所有序列的嵌入
empty_count = 0
total_count = 0

for seq, embedding in embeddings.items():
    total_count += 1
    if len(embedding) == 0:
        print(f"Empty embedding for {seq}")
        empty_count += 1
    else:
        embedding_array = np.array(embedding)
        print(f"{seq}: shape {embedding_array.shape}, sample values {embedding_array[0][:5]}")

print(f"\nSummary: {empty_count} empty embeddings out of {total_count} sequences")

FLPLLAGLAANFLPTIICKISYKC: shape (24, 1152), sample values [-0.00683553  0.00846121 -0.0080218  -0.0046759   0.00282468]
LLAGLAANFLPTIICKISYKC: shape (21, 1152), sample values [-0.01024432  0.0119961  -0.0015966  -0.01145843  0.00279401]
FAGLAANFLPTIICKISYKC: shape (20, 1152), sample values [-0.00392826  0.00806682 -0.00148943 -0.00668949  0.00270863]
FLKLLKKLAAKFLPTIICKISYKC: shape (24, 1152), sample values [-0.00542912  0.01261974 -0.01423866 -0.00896621  0.00155714]
FLKLLKKLAAKLF: shape (13, 1152), sample values [-0.00916339  0.00521027 -0.00639718 -0.00536243  0.00779752]
FLGALFKALSKLL: shape (13, 1152), sample values [-0.0068051   0.00179482 -0.00781735 -0.01049668  0.00905931]
FLKLLAGLLKNFA: shape (13, 1152), sample values [-0.0065772   0.00357459 -0.00946147 -0.00492521  0.01006054]
AIGKFLHSAKKFGKAFVGEIMNS: shape (23, 1152), sample values [-0.01290614  0.01643106 -0.02271784  0.00463184  0.00064991]
GIGKFLHSAKKFAKAFVAEIMNS: shape (23, 1152), sample values [-0.00776426  0.01621407

In [5]:
import json
from Bio import SeqIO

# 合并ACP和nonACP序列
def merge_all_sequences():
    """合并ACP和nonACP序列到一个文件"""
    all_records = []
    
    # 加载ACP序列
    for record in SeqIO.parse("data/source/fasta/ACP.fasta", "fasta"):
        all_records.append(record)
    
    # 加载nonACP序列
    for record in SeqIO.parse("data/source/fasta/nonACP.fasta", "fasta"):
        all_records.append(record)
    
    # 保存合并后的序列
    with open("data/source/fasta/all_sequence/all_sequences.fasta", "w") as output_handle:
        SeqIO.write(all_records, output_handle, "fasta")
    
    print(f"合并完成，共 {len(all_records)} 个序列")

# 运行合并
merge_all_sequences()


合并完成，共 6259 个序列


In [ ]:
# 然后使用合并后的文件重新生成ESMC嵌入
precompute_embeddings(
    fasta_path="data/source/fasta/all_sequence/all_sequences.fasta",
    output_path="data/esmc_embeddings_complete2.json",
    model_size="600m",
    device="cuda"
)

Loading ESMC 600m model...
Model loaded.


Processing sequences:   0%|          | 3/6259 [00:00<04:05, 25.44it/s]

Processing: cancerppd2_1033
Sequence: FLPLLAGLAANFLPTIICKISYKC
Embeddings shape: torch.Size([1, 26, 1152])
Success: Extracted 24 residues for FLPLLAGLAANFLPTIICKISYKC
Processing: cancerppd2_1034
Sequence: LLAGLAANFLPTIICKISYKC
Embeddings shape: torch.Size([1, 23, 1152])
Success: Extracted 21 residues for LLAGLAANFLPTIICKISYKC
Processing: cancerppd2_1035
Sequence: FAGLAANFLPTIICKISYKC
Embeddings shape: torch.Size([1, 22, 1152])
Success: Extracted 20 residues for FAGLAANFLPTIICKISYKC
Processing: cancerppd2_1036
Sequence: FLKLLKKLAAKFLPTIICKISYKC
Embeddings shape: torch.Size([1, 26, 1152])
Success: Extracted 24 residues for FLKLLKKLAAKFLPTIICKISYKC


Processing sequences:   0%|          | 6/6259 [00:00<03:57, 26.32it/s]

Processing: cancerppd2_1038
Sequence: FLKLLKKLAAKLF
Embeddings shape: torch.Size([1, 15, 1152])
Success: Extracted 13 residues for FLKLLKKLAAKLF
Processing: cancerppd2_1039
Sequence: FLGALFKALSKLL
Embeddings shape: torch.Size([1, 15, 1152])
Success: Extracted 13 residues for FLGALFKALSKLL


...

Processing: AntiCPvalid_neg_71
Sequence: KYYGNGVHCGKKTCYVDWGQATASIGKIIVNGWTQHGPWAHR
Embeddings shape: torch.Size([1, 44, 1152])
Success: Extracted 42 residues for KYYGNGVHCGKKTCYVDWGQATASIGKIIVNGWTQHGPWAHR
Processing: MLACP20independent_neg_368
Sequence: MPNWLKKQMQKAFLEKDNYQIKLLNQCWYFYRKKHCS
Embeddings shape: torch.Size([1, 39, 1152])
Success: Extracted 37 residues for MPNWLKKQMQKAFLEKDNYQIKLLNQCWYFYRKKHCS
Processing: AntiCPmaintrain_neg_417
Sequence: FFSLIPSLIGGLVFAIK
Embeddings shape: torch.Size([1, 19, 1152])
Success: Extracted 17 residues for FFSLIPSLIGGLVFAIK
Processing: AntiCPmaintrain_neg_81
Sequence: QESKKGILLKPKTCNTNADCAKFCKGPIQNCLYHTCACVPGNPHCC
Embeddings shape: torch.Size([1, 48, 1152])
Success: Extracted 46 residues for QESKKGILLKPKTCNTNADCAKFCKGPIQNCLYHTCACVPGNPHCC
Processing: AntiCPaltertrain_neg_359
Sequence: PDQDTLKAMMQDAGFESVDYYNLTAGVVALHRGYKFMVEDSQETT
Embeddings shape: torch.Size([1, 47, 1152])
Success: Extracted 45 residues for PDQDTLKAMMQDAGFESVDYYNLTAGVVALHRGYKFMVE

Processing sequences: 100%|██████████| 6259/6259 [04:02<00:00, 25.80it/s]

Processing: MLACP20independent_neg_1041
Sequence: DIRAILSVDGLFDSKA
Embeddings shape: torch.Size([1, 18, 1152])
Success: Extracted 16 residues for DIRAILSVDGLFDSKA
Processing: AntiCPaltertrain_neg_129
Sequence: YSGDDAYATDAILNS
Embeddings shape: torch.Size([1, 17, 1152])
Success: Extracted 15 residues for YSGDDAYATDAILNS
Processing: MLACP20independent_neg_271
Sequence: MPEGPLARGRGRGRGRG
Embeddings shape: torch.Size([1, 19, 1152])
Success: Extracted 17 residues for MPEGPLARGRGRGRGRG
Processing: AntiCPmaintrain_neg_46
Sequence: GIPCGESCVFIPCITGIAGCSCKSKVCYRN
Embeddings shape: torch.Size([1, 32, 1152])
Success: Extracted 30 residues for GIPCGESCVFIPCITGIAGCSCKSKVCYRN
Processing: AntiCPmaintrain_neg_96
Sequence: ATAWDFGPHGLLPIRPIRIRPLCGKDKS
Embeddings shape: torch.Size([1, 30, 1152])
Success: Extracted 28 residues for ATAWDFGPHGLLPIRPIRIRPLCGKDKS
Processing: AntiCPmaintrain_neg_403
Sequence: KLLKWLKKLLK
Embeddings shape: torch.Size([1, 13, 1152])
Success: Extracted 11 residues for KLLKWLKKLL

Embeddings saved to data/esmc_embeddings_complete2.json
Found 0 empty embeddings in saved file


: 